# CommandGemma v1 Training

In [1]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.3 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()

In [4]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# Configuration
MODEL_NAME = "google/gemma-3-270m-it"
DATASET_NAME = "TGoddessana/common-linux-commands-kr-en"
OUTPUT_DIR = "./commandgemma-adapters"
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
MAX_LENGTH = 256
LORA_R = 8
LORA_ALPHA = 16
TEST_SPLIT = 0.1

In [5]:
# Load dataset
print(f"Loading dataset: {DATASET_NAME}")
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Loaded {len(raw_dataset)} samples")

Loading dataset: TGoddessana/common-linux-commands-kr-en


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12250 [00:00<?, ? examples/s]

Loaded 12250 samples


In [6]:
# Prepare dataset
def create_chat_messages(question: str, command: str) -> list[dict]:
    return [
        {"role": "user", "content": question},
        {"role": "assistant", "content": command}
    ]

processed_data = []
for item in raw_dataset:
    korean_q = item.get("korean_question", "")
    english_q = item.get("english_question", "")
    command = item.get("command", "")

    if not command:
        continue

    if korean_q:
        processed_data.append({"messages": create_chat_messages(korean_q, command)})
    if english_q:
        processed_data.append({"messages": create_chat_messages(english_q, command)})

dataset = Dataset.from_list(processed_data)
dataset_splits = dataset.train_test_split(test_size=TEST_SPLIT, shuffle=True, seed=42)
print(f"Train: {len(dataset_splits['train'])}, Test: {len(dataset_splits['test'])}")

Train: 22050, Test: 2450


In [7]:
# Load model with 4-bit quantization
print(f"Loading model: {MODEL_NAME}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.config.pad_token_id = tokenizer.pad_token_id

Loading model: google/gemma-3-270m-it


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [8]:
# LoRA config
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Training config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,
    packing=False,
    optim="adamw_torch",
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=False,
    bf16=True,
)

In [ ]:
# Train
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["test"],
    peft_config=lora_config,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/22050 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/22050 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2450 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2450 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss


In [ ]:
from peft import PeftModel

trainer.push_to_hub("TGoddessana/commandgemma-v1-adapters")
tokenizer.push_to_hub("TGoddessana/commandgemma-v1-adapters")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = peft_model.merge_and_unload()
merged_model.push_to_hub("TGoddessana/commandgemma-v1")
tokenizer.push_to_hub("TGoddessana/commandgemma-v1")

In [ ]:
# Test inference
test_questions = [
    "현재 디렉토리의 파일 목록을 보여줘",
    "Show me all running docker containers",
    "nginx 서비스 재시작해줘"
]

for q in test_questions:
    messages = [{"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=64, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {response}\n")